# Task 3: Clinical Trials in AI Conferences - Proper API Version

This version properly downloads ALL conference paper metadata using APIs with pagination.

## Setup

In [1]:
import pandas as pd
import requests
import re
import time
import json
from typing import List, Dict
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")
print(f"Start time: {datetime.now()}")

Libraries imported
Start time: 2026-03-09 10:49:15.710366


## Configuration

In [2]:
INPUT_FILE = "final_sample_clinical_trials_information.csv"
NEURIPS_CACHE = "neurips_papers_full.json"
ICLR_CACHE = "iclr_papers_full.json"
RESULTS_FILE = "clinical_trials_conference_matches_FINAL.csv"

NEURIPS_START_YEAR = 2008
ICLR_START_YEAR = 2013
END_YEAR = 2024

# Semantic Scholar API
SS_API_KEY = None  # Add your API key here if you have one (optional, but increases rate limits)

print("Config loaded")

Config loaded


## Load Clinical Trials

In [3]:
print("Loading clinical trials...")
df_trials = pd.read_csv(INPUT_FILE, encoding='utf-8-sig', quoting=1, on_bad_lines='skip')
df_trials.columns = df_trials.columns.str.strip()

nct_pattern = re.compile(r'^NCT\d{8}$')
df_trials['nct_id_clean'] = df_trials['nct_id'].astype(str).str.strip()
df_trials = df_trials[df_trials['nct_id_clean'].str.match(nct_pattern, na=False)]

all_nct_ids = df_trials['nct_id_clean'].unique().tolist()
print(f"Loaded {len(all_nct_ids):,} unique NCT IDs")
print(f"Sample NCT IDs: {all_nct_ids[:5]}")

Loading clinical trials...
Loaded 9,428 unique NCT IDs
Sample NCT IDs: ['NCT00175851', 'NCT00359632', 'NCT00415155', 'NCT00422110', 'NCT00422422']


## Download NeurIPS Papers - Proper Method with Pagination

In [4]:
def download_neurips_papers_with_pagination() -> List[Dict]:
    """
    Download NeurIPS papers using proper pagination through Semantic Scholar API.
    """
    print("\n" + "="*80)
    print("Downloading NeurIPS Papers (2008-2024)")
    print("="*80)
    
    all_papers = []
    base_url = "https://api.semanticscholar.org/graph/v1/paper/search"
    
    headers = {}
    if SS_API_KEY:
        headers['x-api-key'] = SS_API_KEY
    
    for year in range(NEURIPS_START_YEAR, END_YEAR + 1):
        print(f"\nYear {year}:")
        year_papers = []
        
        # Try multiple venue variations
        venue_variations = [
            f'venue:NeurIPS year:{year}',
            f'venue:NIPS year:{year}',
            f'venue:"Neural Information Processing Systems" year:{year}'
        ]
        
        for venue_query in venue_variations:
            offset = 0
            limit = 100
            
            while True:
                params = {
                    'query': venue_query,
                    'fields': 'paperId,title,abstract,year,authors,venue,url,citationCount',
                    'limit': limit,
                    'offset': offset
                }
                
                try:
                    response = requests.get(base_url, params=params, headers=headers, timeout=30)
                    
                    if response.status_code == 200:
                        data = response.json()
                        papers = data.get('data', [])
                        total = data.get('total', 0)
                        
                        if not papers:
                            break  # No more papers
                        
                        # Filter to ensure correct venue and year
                        valid_papers = []
                        for p in papers:
                            venue_str = (p.get('venue') or '').lower()
                            paper_year = p.get('year')
                            if ('neurips' in venue_str or 'nips' in venue_str) and paper_year == year:
                                valid_papers.append(p)
                        
                        year_papers.extend(valid_papers)
                        
                        print(f"  Offset {offset}: fetched {len(papers)} papers, {len(valid_papers)} valid (Total: {total})")
                        
                        offset += limit
                        
                        # Stop if we've fetched all available
                        if offset >= total or len(papers) < limit:
                            break
                        
                        time.sleep(1)  # Rate limiting
                    
                    elif response.status_code == 429:
                        print("  Rate limit hit, waiting 30s...")
                        time.sleep(30)
                        continue
                    
                    else:
                        print(f"  Error {response.status_code}: {response.text[:200]}")
                        break
                
                except Exception as e:
                    print(f"  Exception: {e}")
                    break
            
            if year_papers:
                print(f"  Found {len(year_papers)} papers with venue variation: {venue_query}")
                break  # Found papers, no need to try other variations
        
        # Remove duplicates
        seen_ids = set()
        unique_papers = []
        for p in year_papers:
            pid = p.get('paperId')
            if pid and pid not in seen_ids:
                seen_ids.add(pid)
                unique_papers.append(p)
        
        all_papers.extend(unique_papers)
        print(f"  Year {year} total: {len(unique_papers)} unique papers")
    
    print(f"\n{'='*80}")
    print(f"Total NeurIPS papers downloaded: {len(all_papers)}")
    print(f"{'='*80}")
    
    return all_papers

# Check cache first
if Path(NEURIPS_CACHE).exists():
    print(f"Loading NeurIPS papers from cache: {NEURIPS_CACHE}")
    with open(NEURIPS_CACHE, 'r') as f:
        neurips_papers = json.load(f)
    print(f"Loaded {len(neurips_papers)} NeurIPS papers from cache")
else:
    neurips_papers = download_neurips_papers_with_pagination()
    
    # Save cache
    with open(NEURIPS_CACHE, 'w') as f:
        json.dump(neurips_papers, f)
    print(f"Cached NeurIPS papers to: {NEURIPS_CACHE}")



Year 2008:
  Rate limit hit, waiting 30s...


KeyboardInterrupt: 

## Download ICLR Papers - Using OpenReview API

In [ ]:
def download_iclr_papers_with_pagination() -> List[Dict]:
    """
    Download ICLR papers using OpenReview API with proper pagination.
    """
    print("\n" + "="*80)
    print("Downloading ICLR Papers (2013-2024)")
    print("="*80)
    
    all_papers = []
    base_url = "https://api2.openreview.net/notes"
    
    for year in range(ICLR_START_YEAR, END_YEAR + 1):
        print(f"\nYear {year}:")
        year_papers = []
        
        # ICLR venue invitation patterns by year
        invitation_patterns = [
            f'ICLR.cc/{year}/Conference/-/Blind_Submission',
            f'ICLR.cc/{year}/Conference/-/Withdrawn_Submission',
            f'ICLR.cc/{year}/Conference',
            f'ICLR.cc/{year}'
        ]
        
        for invitation in invitation_patterns:
            offset = 0
            limit = 1000
            
            while True:
                params = {
                    'invitation': invitation,
                    'limit': limit,
                    'offset': offset,
                    'details': 'replyCount'
                }
                
                try:
                    response = requests.get(base_url, params=params, timeout=30)
                    
                    if response.status_code == 200:
                        data = response.json()
                        notes = data.get('notes', [])
                        
                        if not notes:
                            break
                        
                        year_papers.extend(notes)
                        print(f"  Offset {offset}: fetched {len(notes)} papers (invitation: {invitation})")
                        
                        offset += limit
                        
                        if len(notes) < limit:
                            break
                        
                        time.sleep(0.5)
                    
                    else:
                        print(f"  Error {response.status_code} for invitation: {invitation}")
                        break
                
                except Exception as e:
                    print(f"  Exception: {e}")
                    break
            
            if year_papers:
                print(f"  Found {len(year_papers)} papers with invitation: {invitation}")
                break
        
        # Remove duplicates
        seen_ids = set()
        unique_papers = []
        for p in year_papers:
            pid = p.get('id')
            if pid and pid not in seen_ids:
                seen_ids.add(pid)
                unique_papers.append(p)
        
        all_papers.extend(unique_papers)
        print(f"  Year {year} total: {len(unique_papers)} unique papers")
    
    print(f"\n{'='*80}")
    print(f"Total ICLR papers downloaded: {len(all_papers)}")
    print(f"{'='*80}")
    
    return all_papers

# Check cache first
if Path(ICLR_CACHE).exists():
    print(f"Loading ICLR papers from cache: {ICLR_CACHE}")
    with open(ICLR_CACHE, 'r') as f:
        iclr_papers = json.load(f)
    print(f"Loaded {len(iclr_papers)} ICLR papers from cache")
else:
    iclr_papers = download_iclr_papers_with_pagination()
    
    # Save cache
    with open(ICLR_CACHE, 'w') as f:
        json.dump(iclr_papers, f)
    print(f"Cached ICLR papers to: {ICLR_CACHE}")

print(f"\nTotal papers downloaded: {len(neurips_papers) + len(iclr_papers):,}")

## Search Through All Papers for NCT IDs

In [ ]:
def search_papers_for_nct_ids(papers: List[Dict], all_nct_ids: List[str], conference: str) -> List[Dict]:
    """
    Search through all papers for NCT ID mentions.
    """
    print(f"\nSearching {len(papers)} {conference} papers for {len(all_nct_ids):,} NCT IDs...")
    
    matches = []
    nct_pattern = re.compile(r'\bNCT\d{8}\b', re.IGNORECASE)
    nct_set = set(nct.lower() for nct in all_nct_ids)
    
    for i, paper in enumerate(papers):
        if (i + 1) % 500 == 0:
            print(f"  Processed {i+1}/{len(papers)} papers, found {len(matches)} matches so far...")
        
        # Extract text based on conference format
        if conference == 'NeurIPS':
            title = paper.get('title', '')
            abstract = paper.get('abstract', '') or ''
            searchable = f"{title} {abstract}".lower()
        else:  # ICLR
            content = paper.get('content', {})
            title = content.get('title', '')
            abstract = content.get('abstract', '') or ''
            searchable = f"{title} {abstract}".lower()
        
        # Find NCT IDs
        found_ncts = nct_pattern.findall(searchable)
        
        for found_nct in found_ncts:
            if found_nct.lower() in nct_set:
                # Create match record
                if conference == 'NeurIPS':
                    match = {
                        'nct_id': found_nct.upper(),
                        'conference': 'NeurIPS',
                        'paper_id': paper.get('paperId'),
                        'paper_title': title,
                        'paper_year': paper.get('year'),
                        'authors': ', '.join([a.get('name', '') for a in paper.get('authors', [])[:5]]),
                        'abstract': abstract[:500],
                        'url': paper.get('url'),
                        'citation_count': paper.get('citationCount'),
                        'match_type': 'nct_id_exact'
                    }
                else:  # ICLR
                    venue = content.get('venue', '')
                    year_match = re.search(r'20\d{2}', venue or paper.get('invitation', ''))
                    paper_year = int(year_match.group()) if year_match else None
                    
                    match = {
                        'nct_id': found_nct.upper(),
                        'conference': 'ICLR',
                        'paper_id': paper.get('id'),
                        'paper_title': title,
                        'paper_year': paper_year,
                        'authors': ', '.join(content.get('authors', [])[:5]),
                        'abstract': abstract[:500],
                        'url': f"https://openreview.net/forum?id={paper.get('forum')}",
                        'citation_count': None,
                        'match_type': 'nct_id_exact'
                    }
                
                matches.append(match)
                print(f"  *** MATCH: {found_nct.upper()} in {conference} {match['paper_year']} - {title[:80]}")
    
    print(f"\nFound {len(matches)} matches in {conference}")
    return matches

# Search both conferences
print("\n" + "="*80)
print("SEARCHING FOR NCT IDs")
print("="*80)

neurips_matches = search_papers_for_nct_ids(neurips_papers, all_nct_ids, 'NeurIPS')
iclr_matches = search_papers_for_nct_ids(iclr_papers, all_nct_ids, 'ICLR')

all_matches = neurips_matches + iclr_matches
print(f"\nTotal matches across both conferences: {len(all_matches)}")

## Enrich with Trial Metadata and Analyze

In [ ]:
if all_matches:
    print("\nEnriching matches with trial metadata...")
    df_matches = pd.DataFrame(all_matches)
    
    # Merge with trial data
    df_merge = df_trials[['nct_id_clean', 'brief_title', 'sponsor_name', 'start_year', 'phase_number']].copy()
    df_merge.columns = ['nct_id', 'trial_title', 'trial_sponsor', 'trial_start_year', 'trial_phase']
    
    df_results = df_matches.merge(df_merge, on='nct_id', how='left')
    
    # Temporal analysis
    df_results['temporal_relationship'] = df_results.apply(
        lambda row: 'Conference Before Trial' if pd.notna(row['paper_year']) and pd.notna(row['trial_start_year']) 
                    and row['paper_year'] < row['trial_start_year']
                    else 'Conference After Trial' if pd.notna(row['paper_year']) and pd.notna(row['trial_start_year'])
                    and row['paper_year'] >= row['trial_start_year']
                    else 'Unknown',
        axis=1
    )
    
    df_results['years_difference'] = df_results.apply(
        lambda row: row['paper_year'] - row['trial_start_year'] 
                    if pd.notna(row['paper_year']) and pd.notna(row['trial_start_year'])
                    else None,
        axis=1
    )
    
    # Save results
    df_results.to_csv(RESULTS_FILE, index=False)
    print(f"Results saved to: {RESULTS_FILE}")
    
    # Summary
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)
    print(f"Total trials searched: {len(all_nct_ids):,}")
    print(f"Total papers searched: {len(neurips_papers) + len(iclr_papers):,}")
    print(f"  - NeurIPS papers: {len(neurips_papers):,}")
    print(f"  - ICLR papers: {len(iclr_papers):,}")
    print(f"\nTotal matches: {len(df_results)}")
    print(f"Unique trials mentioned: {df_results['nct_id'].nunique()}")
    
    print("\n" + "="*40)
    print("By Conference:")
    print(df_results['conference'].value_counts())
    
    print("\n" + "="*40)
    print("By Year:")
    print(df_results['paper_year'].value_counts().sort_index())
    
    print("\n" + "="*40)
    print("Temporal Relationship:")
    print(df_results['temporal_relationship'].value_counts())
    
    print("\n" + "="*40)
    print("Sample Matches:")
    print("="*40)
    display(df_results[['nct_id', 'conference', 'paper_title', 'paper_year', 'trial_title', 'temporal_relationship']].head(20))
    
else:
    print("\nNo matches found.")
    df_results = pd.DataFrame()

## Export Final Dataset

In [ ]:
if len(df_results) > 0:
    final_file = "clinical_trials_ai_conferences_FINAL_DATASET.csv"
    df_results.to_csv(final_file, index=False)
    
    # Create summary
    summary = {
        'execution_date': datetime.now().isoformat(),
        'total_trials_searched': len(all_nct_ids),
        'total_papers_searched': len(neurips_papers) + len(iclr_papers),
        'neurips_papers': len(neurips_papers),
        'iclr_papers': len(iclr_papers),
        'total_matches': len(df_results),
        'unique_trials_mentioned': int(df_results['nct_id'].nunique()),
        'neurips_matches': len(df_results[df_results['conference'] == 'NeurIPS']),
        'iclr_matches': len(df_results[df_results['conference'] == 'ICLR']),
        'conference_before_trial': len(df_results[df_results['temporal_relationship'] == 'Conference Before Trial']),
        'conference_after_trial': len(df_results[df_results['temporal_relationship'] == 'Conference After Trial'])
    }
    
    with open('execution_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n{'='*80}")
    print(f"FINAL DATASET: {final_file}")
    print(f"SUMMARY: execution_summary.json")
    print(f"Total records: {len(df_results)}")
    print(f"{'='*80}")
    print("\nDONE!")
else:
    print("No results to export")